# Exercice 24b : CLIP pré-entraîné, zero-shot puis réglage fin sur images satellite (Colab, GPU T4)

**Chapitre lié** : 24. Vision moderne et multimodalité.

**Objectif.** Prendre un vrai modèle CLIP (ViT-B/32 entraîné sur LAION-2B, 2 milliards de
paires image-texte en anglais, licence MIT, via `open_clip`) et mesurer, sur les 10 classes du jeu
d'images satellite EuroSAT, ce que donnent :

| Étape | Ce qui est appris | Étiquettes utilisées |
|---|---|---|
| A. Zero-shot | rien : on compare l'image à des phrases (« a centered satellite photo of forest. ») | aucune |
| B. Sonde linéaire | une régression logistique sur les plongements d'image figés | 1, 4, 16 par classe, puis les 16 200 |
| C. Réglage fin (fine-tuning) | tout l'encodeur d'image, tête initialisée par les phrases de A | les 16 200 |
| D. Oubli | rien : zero-shot sur CIFAR-10 avant et après C | aucune |

**Perte optimisée** : aucune en A et D (le modèle est utilisé tel quel ; il a été
pré-entraîné avec la perte contrastive InfoNCE, chapitre 23) ; entropie croisée
multiclasse en B et C, $\mathcal{L} = -\frac1N\sum_i \log \hat p_{i,y_i}$.

**Avant de lancer** : Exécution › Modifier le type d'exécution › **GPU T4**, puis
Exécution › Tout exécuter. Durée estimée : 15 à 25 minutes (non mesurée sur T4 au moment
de l'écriture). Coût : 0 €.

In [ ]:
# 1. Vérification du GPU : notebook écrit pour un GPU NVIDIA T4 (Colab gratuit, ou Kaggle « GPU T4 x2 »)
!nvidia-smi
import torch
print("PyTorch", torch.__version__, "| CUDA disponible :", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("ATTENTION : pas de GPU. Exécution › Modifier le type d'exécution › GPU T4, puis tout relancer.\n"
          "Sur CPU, le réglage fin (étape C) prendrait plus d'une heure.")

In [ ]:
# 2. Dépendances : torch et torchvision sont déjà dans l'image Colab (on ne les épingle pas,
# cela réinstallerait ~1 Go). open_clip_torch et timm sont épinglés (versions vérifiées sur
# PyPI pour Python 3.12) ; huggingface_hub, pyarrow, pandas, Pillow et scikit-learn sont
# déjà dans l'image Colab.
!pip install -q open_clip_torch==3.3.0 timm==1.0.30

In [ ]:
# 3. Imports et configuration
import io
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from PIL import Image
from sklearn.linear_model import LogisticRegression

# IA_TEST_LOCAL=1 : petite version (quelques centaines d'images) pour vérifier le code hors Colab
TEST_LOCAL = os.environ.get("IA_TEST_LOCAL") == "1"
SEED = 42
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
FP16 = device == "cuda"   # demi-précision float16 (le T4 ne fait pas de bfloat16)

def seed_everything(graine=SEED):
    random.seed(graine)
    np.random.seed(graine)
    torch.manual_seed(graine)
    torch.cuda.manual_seed_all(graine)

seed_everything()
config = dict(
    n_train=320 if TEST_LOCAL else None,    # None = les 16 200 images d'entraînement d'EuroSAT
    n_test=200 if TEST_LOCAL else None,     # None = les 5 400 images de test
    n_cifar=200 if TEST_LOCAL else 2000,    # images de test CIFAR-10 pour mesurer l'oubli (étape D)
    epoques=1 if TEST_LOCAL else 2,
    lot=32 if TEST_LOCAL else 128,
    lr=1e-5, weight_decay=0.1, echauffement=0.1,   # réglage fin : taux faible (voir questions)
    shots=(1, 4, 16),                               # sondes linéaires à peu d'exemples
)
print("Périphérique :", device, "| fp16 :", FP16, "| test local :", TEST_LOCAL)
print(config)

## Le modèle : CLIP ViT-B/32 de LAION

La bibliothèque libre `open_clip` charge l'architecture CLIP et des poids entraînés sur
LAION-2B, le sous-ensemble anglais (2 milliards de paires image-texte) du jeu public
LAION-5B. Le dépôt
`laion/CLIP-ViT-B-32-laion2B-s34B-b79K` est sous licence MIT et ne demande ni compte ni
token. Deux encodeurs : un **ViT-B/32** pour l'image (image 224×224 découpée en
7×7 = 49 patchs de 32×32 pixels) et un **Transformer de texte** ; tous deux projettent dans
le même espace de dimension 512.

In [ ]:
# 4. Chargement de CLIP (≈ 600 Mo, téléchargés une fois dans le cache Hugging Face)
modele, pretraitement_train, pretraitement = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k")
tokeniseur = open_clip.get_tokenizer("ViT-B-32")
modele = modele.to(device).eval()
n_img = sum(p.numel() for p in modele.visual.parameters())
n_txt = sum(p.numel() for n, p in modele.named_parameters() if not n.startswith("visual."))
print(f"Encodeur d'image : {n_img / 1e6:.1f} M paramètres ; encodeur de texte (et le reste) : {n_txt / 1e6:.1f} M")
print("Température apprise : échelle des logits =", round(modele.logit_scale.exp().item(), 1))
print(pretraitement)

## Données : EuroSAT (images Sentinel-2) et CIFAR-10

**EuroSAT** (Helber et al., 2019) : 27 000 images satellite 64×64 de 10 classes
d'occupation du sol, copie Hugging Face `timm/eurosat-rgb` (licence MIT, ~92 Mo, découpage
16 200 / 5 400 / 5 400). Les photos satellite sont sans doute peu présentes dans les légendes du web : c'est
un cas où le zero-shot de CLIP est faible, donc où l'on voit bien ce qu'apportent quelques
étiquettes.

**CIFAR-10** (test seulement, 2 000 images) sert à l'étape D : le réglage fin sur des images
satellite dégrade-t-il ce que le modèle savait faire ailleurs ?

In [ ]:
# 5. Téléchargement des fichiers parquet et décodage des images (PIL)
def lire_parquet(depot, fichier, n=None, colonne_image="image", colonne_label="label"):
    chemin = hf_hub_download(depot, fichier, repo_type="dataset")
    table = pd.read_parquet(chemin)
    if n:
        table = table.sample(n=n, random_state=SEED)   # sous-ensemble tiré au hasard, graine fixe
    images = [Image.open(io.BytesIO(d["bytes"])).convert("RGB") for d in table[colonne_image]]
    return images, np.array(table[colonne_label])

euro_train, y_train = lire_parquet("timm/eurosat-rgb", "data/train-00000-of-00001.parquet", config["n_train"])
euro_test, y_test = lire_parquet("timm/eurosat-rgb", "data/test-00000-of-00001.parquet", config["n_test"])
cifar_test, y_cifar = lire_parquet("uoft-cs/cifar10", "plain_text/test-00000-of-00001.parquet",
                                   config["n_cifar"], colonne_image="img", colonne_label="label")
CLASSES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
           "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]
CLASSES_CIFAR = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
print(f"EuroSAT : {len(euro_train)} images d'entraînement, {len(euro_test)} de test ; "
      f"CIFAR-10 : {len(cifar_test)} images de test")
print("Effectifs par classe (entraînement) :", np.bincount(y_train, minlength=10))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axes.flat):
    i = int(np.flatnonzero(y_train == k)[0])
    ax.imshow(euro_train[i]); ax.set_title(CLASSES[k], fontsize=9); ax.axis("off")
fig.suptitle("Une image EuroSAT (64×64) par classe")
plt.show()

In [ ]:
# 6. Encodage des images par le ViT figé (plongements normalisés), par lots
@torch.no_grad()
def encoder_images(images, lot=256):
    modele.eval()
    sorties = []
    for i in range(0, len(images), lot):
        x = torch.stack([pretraitement(im) for im in images[i:i + lot]]).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
            z = modele.encode_image(x)
        sorties.append(F.normalize(z.float(), dim=-1).cpu())
    return torch.cat(sorties)

t0 = time.perf_counter()
Z_test = encoder_images(euro_test)
Z_cifar_avant = encoder_images(cifar_test)
print(f"{len(euro_test) + len(cifar_test)} images encodées en {time.perf_counter() - t0:.1f} s ;"
      f" plongements de dimension {Z_test.shape[1]}")

## A. Zero-shot : classer avec des phrases

Pour chaque classe, on encode une ou plusieurs phrases ; le « classifieur » est la matrice
de leurs plongements normalisés. Une image est rangée dans la classe dont la phrase a le
plus grand cosinus avec elle. Quatre façons d'écrire les phrases :

1. le nom brut de la classe (`AnnualCrop`) ;
2. un nom lisible dans un gabarit (`a centered satellite photo of annual crop land.`) ;
3. trois gabarits moyennés (ceux du dépôt officiel de CLIP pour EuroSAT) ;
4. les mêmes idées **en français** (`une photo satellite de cultures annuelles.`).

In [ ]:
# 7. Zero-shot avec quatre jeux de phrases
NOMS_LISIBLES = {   # noms et gabarits du fichier data/prompts.md du dépôt openai/CLIP (section EuroSAT)
    "AnnualCrop": "annual crop land", "Forest": "forest",
    "HerbaceousVegetation": "brushland or shrubland", "Highway": "highway or road",
    "Industrial": "industrial buildings or commercial buildings", "Pasture": "pasture land",
    "PermanentCrop": "permanent crop land",
    "Residential": "residential buildings or homes or apartments",
    "River": "river", "SeaLake": "lake or sea",
}
NOMS_FR = {
    "AnnualCrop": "cultures annuelles", "Forest": "forêt",
    "HerbaceousVegetation": "broussailles", "Highway": "autoroute ou route",
    "Industrial": "bâtiments industriels ou commerciaux", "Pasture": "pâturages",
    "PermanentCrop": "cultures permanentes", "Residential": "quartier résidentiel",
    "River": "rivière", "SeaLake": "lac ou mer",
}
GABARITS = ["a centered satellite photo of {}.", "a centered satellite photo of a {}.",
            "a centered satellite photo of the {}."]

@torch.no_grad()
def classifieur_texte(phrases_par_classe):
    # phrases_par_classe : une liste de phrases par classe ; on moyenne leurs plongements
    poids = []
    for phrases in phrases_par_classe:
        z = modele.encode_text(tokeniseur(phrases).to(device)).float()
        z = F.normalize(z, dim=-1).mean(0)
        poids.append(F.normalize(z, dim=0))
    return torch.stack(poids).cpu()          # (classes, 512)

def exactitude(Z, W, y):
    return ((Z @ W.T).argmax(1).numpy() == y).mean()

jeux = {
    "nom brut": [[c] for c in CLASSES],
    "un gabarit": [[GABARITS[0].format(NOMS_LISIBLES[c])] for c in CLASSES],
    "3 gabarits": [[g.format(NOMS_LISIBLES[c]) for g in GABARITS] for c in CLASSES],
    "français": [[f"une photo satellite de {NOMS_FR[c]}."] for c in CLASSES],
}
resultats = {"zero_shot": {}}
for nom, phrases in jeux.items():
    W = classifieur_texte(phrases)
    resultats["zero_shot"][nom] = float(exactitude(Z_test, W, y_test))
    print(f"zero-shot, {nom:11s}: {resultats['zero_shot'][nom]:.1%}   ex. « {phrases[0][0]} »")
W_zero = classifieur_texte(jeux["3 gabarits"])
print("Hasard : 10 %")

## B. Sonde linéaire : quelques étiquettes sur des plongements figés

On garde l'encodeur d'image figé et on entraîne une **régression logistique** (chapitre 01)
sur ses plongements, avec 1, 4 ou 16 images étiquetées par classe, puis avec tout le jeu
d'entraînement. Combien d'exemples étiquetés « vaut » le zero-shot ?

In [ ]:
# 8. Sondes linéaires (scikit-learn, entropie croisée + pénalité L2)
t0 = time.perf_counter()
Z_train = encoder_images(euro_train)
print(f"{len(euro_train)} images d'entraînement encodées en {time.perf_counter() - t0:.1f} s")

def sonde(indices):
    clf = LogisticRegression(C=10.0, max_iter=2000)
    clf.fit(Z_train[indices].numpy(), y_train[indices])
    return float(clf.score(Z_test.numpy(), y_test))

rng = np.random.default_rng(SEED)
resultats["sonde"] = {}
for k in config["shots"]:
    # k images par classe, tirées au hasard ; moyenne sur 3 tirages (le résultat varie beaucoup)
    scores = []
    for _ in range(3):
        idx = np.concatenate([rng.choice(np.flatnonzero(y_train == c), k, replace=False) for c in range(10)])
        scores.append(sonde(idx))
    resultats["sonde"][f"{k} par classe"] = float(np.mean(scores))
    print(f"sonde linéaire, {k:2d} image(s) par classe : {np.mean(scores):.1%} (3 tirages : "
          + ", ".join(f"{s:.1%}" for s in scores) + ")")
resultats["sonde"]["tout le jeu"] = sonde(np.arange(len(y_train)))
print(f"sonde linéaire, {len(y_train)} images : {resultats['sonde']['tout le jeu']:.1%}")

## C. Réglage fin (fine-tuning) de l'encodeur d'image

On entraîne **tout l'encodeur d'image** (88 M paramètres) avec une tête linéaire 512 → 10
**initialisée par les phrases de l'étape A** (les plongements des trois gabarits,
multipliés par l'échelle des logits de CLIP) : au départ, le modèle réglé fait exactement
le zero-shot, puis il s'en écarte. Taux faible (10⁻⁵), AdamW, échauffement puis cosinus,
float16 avec `GradScaler`. L'encodeur de texte n'est pas modifié.

In [ ]:
# 9. Réglage fin
class ClassifieurClip(torch.nn.Module):
    def __init__(self, visuel, W, echelle):
        super().__init__()
        self.visuel = visuel
        self.tete = torch.nn.Linear(W.shape[1], W.shape[0], bias=True)
        with torch.no_grad():   # tête = phrases du zero-shot × échelle de CLIP
            self.tete.weight.copy_(W * echelle)
            self.tete.bias.zero_()
    def forward(self, x):
        return self.tete(F.normalize(self.visuel(x), dim=-1))

classif = ClassifieurClip(modele.visual, W_zero.to(device), modele.logit_scale.exp().item()).to(device)

class Images(torch.utils.data.Dataset):
    # Pré-traitement appliqué image par image (recadrage aléatoire tiré pour chaque image)
    def __init__(self, images, y, transform):
        self.images, self.y, self.transform = images, y, transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        return self.transform(self.images[i]), int(self.y[i])

train_dl = torch.utils.data.DataLoader(
    Images(euro_train, y_train, pretraitement_train), batch_size=config["lot"], shuffle=True,
    num_workers=0 if TEST_LOCAL else 2, drop_last=True,
    generator=torch.Generator().manual_seed(SEED))

optim = torch.optim.AdamW(classif.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
total = config["epoques"] * len(train_dl)
echauffement = max(1, int(config["echauffement"] * total))
sched = torch.optim.lr_scheduler.LambdaLR(
    optim, lambda s: min(1.0, (s + 1) / echauffement) * 0.5 * (1 + np.cos(np.pi * min(1.0, s / total))))
scaler = torch.amp.GradScaler("cuda", enabled=FP16)

@torch.no_grad()
def exactitude_classif(images, y, lot=256):
    classif.eval()
    bons = 0
    for i in range(0, len(images), lot):
        x = torch.stack([pretraitement(im) for im in images[i:i + lot]]).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
            bons += (classif(x).argmax(1).cpu().numpy() == y[i:i + lot]).sum()
    return float(bons / len(y))

print(f"Avant réglage fin (doit égaler le zero-shot à 3 gabarits) : {exactitude_classif(euro_test, y_test):.1%}")
historique = {"pas": [], "perte": [], "exactitude": []}
t0, pas = time.perf_counter(), 0
for ep in range(config["epoques"]):
    classif.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
            perte = F.cross_entropy(classif(x), y)
        optim.zero_grad(set_to_none=True)
        scaler.scale(perte).backward()
        scaler.step(optim)
        scaler.update()
        sched.step()
        pas += 1
        historique["pas"].append(pas)
        historique["perte"].append(perte.item())
    historique["exactitude"].append(exactitude_classif(euro_test, y_test))
    print(f"époque {ep + 1}/{config['epoques']} : perte {np.mean(historique['perte'][-len(train_dl):]):.3f}, "
          f"exactitude en test {historique['exactitude'][-1]:.1%} ({time.perf_counter() - t0:.0f} s)")
resultats["reglage_fin"] = historique["exactitude"][-1]
resultats["duree_reglage_fin_s"] = time.perf_counter() - t0

## D. Ce que le réglage fin a coûté ailleurs

L'encodeur d'image a changé, l'encodeur de texte non. On refait le zero-shot de CLIP sur
2 000 images de CIFAR-10 (« a photo of a cat. »…) avec l'encodeur d'image **réglé** et on
compare au même calcul fait avant le réglage.

In [ ]:
# 10. Zero-shot CIFAR-10 avant / après le réglage fin
W_cifar = classifieur_texte([[f"a photo of a {c}."] for c in CLASSES_CIFAR])
Z_cifar_apres = encoder_images(cifar_test)   # modele.visual est l'encodeur réglé (même objet)
resultats["cifar_avant"] = float(exactitude(Z_cifar_avant, W_cifar, y_cifar))
resultats["cifar_apres"] = float(exactitude(Z_cifar_apres, W_cifar, y_cifar))
print(f"CIFAR-10 zero-shot : {resultats['cifar_avant']:.1%} avant, {resultats['cifar_apres']:.1%} après réglage fin")

In [ ]:
# 11. Bilan, figure et seuils (vérifiés seulement en version complète)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
noms = list(resultats["zero_shot"]) + list(resultats["sonde"]) + ["réglage fin"]
vals = list(resultats["zero_shot"].values()) + list(resultats["sonde"].values()) + [resultats["reglage_fin"]]
couleurs = ["#0072B2"] * len(resultats["zero_shot"]) + ["#E69F00"] * len(resultats["sonde"]) + ["#009E73"]
barres = ax1.barh(noms[::-1], [100 * v for v in vals[::-1]], color=couleurs[::-1])
ax1.bar_label(barres, fmt="%.1f %%", fontsize=8)
ax1.axvline(10, color="gray", ls=":", label="hasard")
ax1.set(xlabel="exactitude en test EuroSAT (%)", xlim=(0, 110),
        title="Zero-shot (bleu), sonde linéaire (orange), réglage fin (vert)")
ax1.legend(loc="lower right")
ax2.bar(["avant", "après"], [100 * resultats["cifar_avant"], 100 * resultats["cifar_apres"]],
        color=["#0072B2", "#D55E00"])
ax2.set(ylabel="exactitude zero-shot CIFAR-10 (%)", ylim=(0, 100),
        title="CIFAR-10 avant / après réglage fin sur EuroSAT")
fig.tight_layout()
fig.savefig("clip_24b.png", dpi=150)
plt.show()

with open("resultats_24b.json", "w") as f:
    json.dump(resultats, f, indent=2, ensure_ascii=False)
print(json.dumps(resultats, indent=2, ensure_ascii=False))

if not TEST_LOCAL:
    # Seuils prudents, non mesurés sur T4 au moment de l'écriture (voir le README)
    assert max(resultats["zero_shot"].values()) >= 0.35, "zero-shot anormalement bas"
    assert resultats["sonde"]["tout le jeu"] >= 0.90, "sonde linéaire anormalement basse"
    assert resultats["reglage_fin"] >= 0.95, "réglage fin anormalement bas"
    print("Seuils atteints.")

In [ ]:
# 12. Sauvegarder les résultats avant la fin de la session (Colab : téléchargement ;
# Kaggle : les fichiers restent dans /kaggle/working, onglet Output)
try:
    from google.colab import files
    for nom in ("resultats_24b.json", "clip_24b.png"):
        files.download(nom)
except ImportError:
    print("Hors Colab : fichiers écrits dans", os.getcwd())